# 06 – Few-label cross-dataset adaptation (pair-disjoint)
**Reproduces:** Section 5.7 (exploratory few-label target adaptation: the source-only, frozen-feature residual Ridge and target-only fingerprint Ridge rows of the few-label table) and Supplementary Section S8

**Responds to:** Reviewer 1 Comment 2 (additional exploratory few-label adaptation); also cited in Reviewer 3 Comment 3 and in the limitations (Section 5.9)

**What it does:** adapts the three trained source models of each cross-dataset direction (DrugComb → O'Neil and O'Neil → DrugComb, common Loewe labels, seeds 42–44) with a small number of labelled target records. Budgets: 16, 32, 64 and 128 labelled records (primary: 64, one record per support pair); five support draws (101–105); one fixed query set that shares no drug pair with the support sets. Source models are not retrained, and no query label is used for fitting or for any selection. This is supervised target adaptation: it is neither independent zero-shot transfer nor a cold-drug evaluation after adaptation.

**How to run:** `Runtime → Run all` on a GPU runtime (T4 recommended for the feature export in Cell 4). The helper modules (`fewshot_core.py`, `export_features.py`, `source_runtime.py`, `test_fewshot.py`) must be in `DualGATSynergy/fewshot_tools/` on Drive; Cell 1 stops with a message if they are missing. The project folder `DualGATSynergy/` on Drive must also contain the trained cross-dataset checkpoints (`cross_dataset/`). Results are written to `fewshot_extension_v1/` and never overwrite the original results.

**Reading the output:** SDs are descriptive (they mix source seeds and support draws and are not independent runs). Bootstrap intervals resample query pairs, ignore the dependence between pairs that share a drug, and are exploratory.

In [ ]:
# ===== Cell 1: Load the helper modules from the project folder =====
# The helper modules (fewshot_core.py, export_features.py, source_runtime.py, test_fewshot.py)
# are shipped in DualGATSynergy/fewshot_tools/.
import sys, pathlib
from google.colab import files, drive      # `files` is used again in Cell 7
drive.mount('/content/drive')
TOOLS = pathlib.Path('/content/drive/MyDrive/DualGATSynergy/fewshot_tools')
needed = {'fewshot_core.py', 'export_features.py', 'source_runtime.py', 'test_fewshot.py'}
assert all((TOOLS/name).exists() for name in needed), f'Copy the helper modules into {TOOLS}'
sys.path.insert(0, str(TOOLS))

Mounted at /content/drive


In [ ]:
# ===== Cell 2: Environment and configuration =====
# Use the same environment/requirements as the original source checkpoints.
# Colab already provides PyTorch; do not replace it unnecessarily.
!pip -q install rdkit torch-geometric transformers scikit-learn scipy pandas matplotlib pubchempy xgboost
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/DualGATSynergy')
CHECKPOINT_DIR = ROOT/'cross_dataset' # or ROOT/'cross_dataset_retrained'
SEEDS = (42,43,44)
BUDGETS = (16,32,64,128)
DRAWS = (101,102,103,104,105)
PRIMARY_BUDGET = 64
SPLIT_SEED = 2027
OUTPUT = ROOT/'fewshot_extension_v1'
assert ROOT.exists(), 'Set ROOT to the original project folder.'


## Self-test on synthetic data
Passing these tests checks the software only. It does not validate the exporter on the real checkpoints and says nothing about biomedical performance.

In [ ]:
# ===== Cell 3: Self-test on synthetic data and environment record =====
import runpy, subprocess, json, platform
runpy.run_path(str(TOOLS/'test_fewshot.py'), run_name='__main__')
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT/'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True));


PASS: pair disjointness, exact/nested label budgets, query-label isolation, constant-shift PCC, deterministic rerun, protocol lock, paired bootstrap, end-to-end synthetic outputs.


## Export frozen representations and attention diagnostics
The cell first checks every data and checkpoint path and only then reads the large source CSV. It reconstructs the original data and source split without re-running any earlier training or evaluation, purges every source pair that overlaps with the target, and runs inference in eval mode. Source predictions are exactly order-averaged; the adapter features use the sum and the absolute difference. New artifacts are written separately from the original results.

*Requirement:* the source checkpoints must be present. Saved predictions alone are not enough, because the latent features can only be extracted with the trained weights; a randomly initialised model must not be substituted.

In [ ]:
# ===== Cell 4: Export the frozen features of the source models =====
from export_features import export_all
BUNDLES = export_all(ROOT, checkpoint_dir=CHECKPOINT_DIR, seeds=SEEDS, batch_size=128)
print('Feature bundles:', len(BUNDLES))



  source -> train: 38748 | validation: 11901 | source drugs: 274 (validation: 41)
  target -> 10494 of 22737 rows (46.2%) after the cell-line filter | 18 of 39 cell lines shared (2332 rows matched after name normalization)
  target -> entirely new combinations: 10463 (99.7%) | at least one unseen drug: 9900 (94.3%) | both unseen: 5904 (56.3%)
Valid samples: 10,170 of 10,170
EXPORTED /content/drive/MyDrive/DualGATSynergy/fewshot_extension_v1/DrugComb_to_ONeil_seed42.npz
EXPORTED /content/drive/MyDrive/DualGATSynergy/fewshot_extension_v1/DrugComb_to_ONeil_seed43.npz
EXPORTED /content/drive/MyDrive/DualGATSynergy/fewshot_extension_v1/DrugComb_to_ONeil_seed44.npz
  source -> train: 16497 | validation: 6240 | source drugs: 38 (validation: 5)
  target -> 6569 of 50719 rows (13.0%) after the cell-line filter | 21 of 228 cell lines shared (2073 rows matched after name normalization)
  target -> entirely new combinations: 6538 (99.5%) | at least one unseen drug: 6529 (99.4%) | both unseen: 5323

## Fit the target adapters (support-only model selection)
No query label enters the fitting or the choice of the regularisation strength (alpha); every label used for inner validation is counted in the labelled budget. The "residual-independent" variant still starts from an IMCA-containing source predictor: it ablates the *adapter features*, not the IMCA architecture of the source model.

In [ ]:
# ===== Cell 5: Fit the adapters and summarize =====
import pandas as pd
from fewshot_core import run_bundle, lock_manifest
lock_manifest(OUTPUT/'analysis_plan.json', dict(primary_budget=PRIMARY_BUDGET,
    primary_comparison=['residual_full_features','source_only_sym'],
    essential_control='target_only_fingerprint_ridge',
    secondary_adapter_ablation=['residual_full_features','residual_independent_features'],
    primary_metric='pcc',secondary_metrics=['within_cell_pcc','scc','ap'],
    scope='Exploratory pair-disjoint supervised target adaptation, shared source-observed cells.'))
all_metrics=[]
for name in BUNDLES:
    p=Path(name)
    m=run_bundle(p,OUTPUT/(p.stem+'_adaptation'),budgets=BUDGETS,draws=DRAWS,split_seed=SPLIT_SEED)
    m['bundle']=p.stem
    m['seed']=int(p.stem.rsplit('seed',1)[1])
    m['direction']=p.stem.rsplit('_seed',1)[0]
    all_metrics.append(m)
metrics=pd.concat(all_metrics,ignore_index=True)
metrics.to_csv(OUTPUT/'all_metrics.csv',index=False)
# Descriptive, not independent 15-run testing or pooled prediction ensembling.
summary=metrics.groupby(['direction','budget','method']).agg(
    pcc_mean=('pcc','mean'),pcc_run_sd=('pcc','std'),scc_mean=('scc','mean'),
    within_cell_pcc=('within_cell_pcc','mean'),ap_mean=('ap','mean'),auc_mean=('auc','mean'))
summary.to_csv(OUTPUT/'all_descriptive_summary.csv')
display(summary)


pcc_mean  \
direction         budget method                                            
DrugComb_to_ONeil 16     residual_full_features                 0.112909   
                         residual_independent_features          0.119385   
                         source_only_sym                        0.116392   
                         source_uniform_attention_intervention  0.116807   
                         support_mean_residual                  0.116392   
                         target_head_frozen_full                0.071024   
                         target_only_fingerprint_ridge          0.125801   
                  32     residual_full_features                 0.152713   
                         residual_independent_features          0.148501   
                         source_only_sym                        0.116392   
                         source_uniform_attention_intervention  0.116807   
                         support_mean_residual                  0.116392   
                         target_head_frozen_full                0.132462   
                         target_only_fingerprint_ridge          0.205104   
                  64     residual_full_features                 0.213357   
                         residual_independent_features          0.204827   
                         source_only_sym                        0.116392   
                         source_uniform_attention_intervention  0.116807   
                         support_mean_residual                  0.116392   
                         target_head_frozen_full                0.209773   
                         target_only_fingerprint_ridge          0.244661   
                  128    residual_full_features                 0.246150   
                         residual_independent_features          0.242420   
                         source_only_sym                        0.116392   
                         source_uniform_attention_intervention  0.116807   
                         support_mean_residual                  0.116392   
                         target_head_frozen_full                0.249460   
                         target_only_fingerprint_ridge          0.318487   
ONeil_to_DrugComb 16     residual_full_features                 0.097124   
                         residual_independent_features          0.104409   
                         source_only_sym                        0.128686   
                         source_uniform_attention_intervention  0.127493   
                         support_mean_residual                  0.128686   
                         target_head_frozen_full                0.051922   
                         target_only_fingerprint_ridge          0.099941   
                  32     residual_full_features                 0.113379   
                         residual_independent_features          0.110052   
                         source_only_sym                        0.128686   
                         source_uniform_attention_intervention  0.127493   
                         support_mean_residual                  0.128686   
                         target_head_frozen_full                0.084882   
                         target_only_fingerprint_ridge          0.145794   
                  64     residual_full_features                 0.120045   
                         residual_independent_features          0.122422   
                         source_only_sym                        0.128686   
                         source_uniform_attention_intervention  0.127493   
                         support_mean_residual                  0.128686   
                         target_head_frozen_full                0.117723   
                         target_only_fingerprint_ridge          0.178496   
                  128    residual_full_features                 0.158427   
                         residual_independent_features          0.157327   
                         source_only

In [ ]:
import pandas as pd
summary = pd.read_csv(OUTPUT/'all_descriptive_summary.csv')          # written by Cell 5
PAPER = {   # few-label table at 64 labels: (DrugComb->O'Neil, O'Neil->DrugComb)
    'source_only_sym':               (0.116, 0.129),
    'residual_full_features':        (0.213, 0.120),
    'target_only_fingerprint_ridge': (0.245, 0.178)}
d = summary[summary.budget == PRIMARY_BUDGET].set_index(['method', 'direction'])['pcc_mean']
for m, (a, b) in PAPER.items():
    x, y = d[(m, 'DrugComb_to_ONeil')], d[(m, 'ONeil_to_DrugComb')]
    ok = abs(x - a) < 1.5e-3 and abs(y - b) < 1.5e-3
    print(f"{'✓' if ok else '⚠'} {m:31s} DrugComb->O'Neil {x:.3f} (paper: {a:.3f}) | O'Neil->DrugComb {y:.3f} (paper: {b:.3f})")

✓ source_only_sym                 DrugComb->O'Neil 0.116 (paper: 0.116) | O'Neil->DrugComb 0.129 (paper: 0.129)
✓ residual_full_features          DrugComb->O'Neil 0.213 (paper: 0.213) | O'Neil->DrugComb 0.120 (paper: 0.120)
✓ target_only_fingerprint_ridge   DrugComb->O'Neil 0.245 (paper: 0.245) | O'Neil->DrugComb 0.178 (paper: 0.178)


## Paired uncertainty on the fixed query
The same pair-cluster resample is shared across all source seeds and support draws. These exploratory intervals ignore the dependence between pairs that share a drug; they are not evidence of population-wide superiority, and the support draws are not independent datasets.

In [ ]:
# ===== Cell 6: Paired bootstrap at the primary budget (64 labels) =====
from fewshot_core import paired_pair_bootstrap
comparisons=[]
for direction in metrics.direction.unique():
    chunks=[]
    query_ids=None
    for seed in SEEDS:
        stem=f'{direction}_seed{seed}'
        pred=pd.read_csv(OUTPUT/(stem+'_adaptation')/'query_predictions.csv.gz',dtype={'row_id':str,'pair_key':str})
        ids=set(pred.row_id)
        if query_ids is None: query_ids=ids
        assert query_ids==ids, 'Seeds have different query sets: do not combine as paired runs.'
        pred['seed']=seed; chunks.append(pred)
    pred=pd.concat(chunks,ignore_index=True)
    for comparator in ['source_only_sym','target_only_fingerprint_ridge','residual_independent_features']:
        result=paired_pair_bootstrap(pred,method_a='residual_full_features',method_b=comparator,
                                     budget=PRIMARY_BUDGET,n_boot=2000)
        comparisons.append(dict(direction=direction,**result))
comparison=pd.DataFrame(comparisons)
comparison.to_csv(OUTPUT/'paired_primary_budget_comparisons.csv',index=False)
display(comparison)


,direction,method_a,method_b,budget,delta_pcc,ci_low,ci_high,n_pairs,warning
0,DrugComb_to_ONeil,residual_full_features,source_only_sym,64,0.096965,0.055507,0.139737,283,"Exploratory; fixed query, few source seeds; sh..."
1,DrugComb_to_ONeil,residual_full_features,target_only_fingerprint_ridge,64,-0.031303,-0.071843,0.005392,283,"Exploratory; fixed query, few source seeds; sh..."
2,DrugComb_to_ONeil,residual_full_features,residual_independent_features,64,0.008530,0.003154,0.013638,283,"Exploratory; fixed query, few source seeds; sh..."
3,ONeil_to_DrugComb,residual_full_features,source_only_sym,64,-0.008641,-0.034331,0.016297,458,"Exploratory; fixed query, few source seeds; sh..."
4,ONeil_to_DrugComb,residual_full_features,target_only_fingerprint_ridge,64,-0.058451,-0.100838,-0.016731,458,"Exploratory; fixed query, few source seeds; sh..."
5,ONeil_to_DrugComb,residual_full_features,residual_independent_features,64,-0.002377,-0.004061,-0.000707,458,"Exploratory; fixed query, few source seeds; sh..."


## Collect the outputs
The output ZIP contains the metrics, the query predictions, the support manifests and the attention audits. It excludes the original raw data, the checkpoint weights and the large frozen-feature `.npz` files. Adapted results are reported separately from, and never replace, the independent cross-dataset results (notebook 04). Inconclusive and negative results are part of the output.

In [ ]:
# ===== Cell 7: Package the outputs =====
import zipfile
archive_path=Path('/content/fewshot_outputs.zip')
with zipfile.ZipFile(archive_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT.rglob('*'):
        if p.is_file() and p.suffix!='.npz':
            z.write(p,str(p.relative_to(OUTPUT)))
files.download(str(archive_path))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>